## 2장 1강: INSERT와 샘플 데이터 구성

### 2장 실습 스키마 준비

```
drop table if exists query_logs;
drop table if exists documents;
drop table if exists users;

create table users (
  id bigint generated always as identity primary key,
  email text unique not null,
  name text not null,
  created_at timestamptz not null default now()
);

create table documents (
  id bigint generated always as identity primary key,
  user_id bigint not null references users(id),
  title text not null,
  content text,
  status text not null default 'draft' check (status in ('draft', 'published', 'archived')),
  is_deleted boolean not null default false,
  created_at timestamptz not null default now(),
  updated_at timestamptz not null default now()
);

create table query_logs (
  id bigint generated always as identity primary key,
  user_id bigint not null references users(id),
  document_id bigint references documents(id),
  question text not null,
  response text,
  created_at timestamptz not null default now()
);
```

### 샘플 데이터 입력과 검증

```
TRUNCATE TABLE 테이블명 RESTART IDENTITY CASCADE;
-- 1) 사용자
INSERT INTO users (email, name)
VALUES
  ('minji@example.com', '김민지'),
  ('junho@example.com', '이준호');

-- 2) 문서 (user_id는 위에서 생성된 id를 참조)
INSERT INTO documents (user_id, title, content, status)
VALUES
  (1, 'AI 서비스 기획서', '기획 초안입니다.', 'draft'),
  (1, 'RAG 파이프라인 설계', '검색 증강 생성 설명', 'published'),
  (2, '데이터베이스 정리 노트', '테이블 설계 메모', 'draft');

```

-- 3) 로그
```
INSERT INTO query_logs (user_id, document_id, question, response)
VALUES (
  1,
  2,
  'RAG 파이프라인은 어떻게 구성하나요?',
  '문서 임베딩 → 벡터 검색 → 프롬프트 생성 순서로 구성합니다.'
);
```